<font size="+3"><strong>データの準備・変換</strong></font>

# 必要なライブラリのインポート


In [1]:
# 汎用ライブラリのimport
import os
import sys
import copy
import warnings
import textwrap
from operator import itemgetter
import logging

import math
import random

import platform
import locale
import gzip
import io
import hashlib
import json
import pickle

import pandas as pd
import numpy as np
from scipy.stats import chisquare

import sylib

# %config Completer.use_jedi = False
# %matplotlib inline

progress_bar = sylib.utils.ProgressBar()

# Jupyterでlog出力するための特殊な設定
# StreamHandlerの出力先としてsys.stderrを指定している。
logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler]
)
logger=logging.getLogger()

print("python = %d.%d.%d" % sys.version_info[:3])
print("pandas = %s" % pd.__version__)
print("numpy = %s" % np.__version__)
print("sylib = %s" % sylib.__version__)


python = 3.11.15
pandas = 3.0.3
numpy = 2.4.6
sylib = 0.3.0.dev0+ae18bb2


## 作業ディレクトの設定
普通にinputやoutputファイルの絶対パスを入力してもいい

In [2]:
%ls
current_dir = %pwd
print(current_dir)


 ドライブ D のボリューム ラベルがありません。D:\S_Yamasaki\Projects\Academic\PolysomeFLcDNAseq__A_thaliana\T87-NC-HS\Analysis.FujimakiData.1.4-Takazawa

 ボリューム シリアル番号は 26C3-94B7 です

 D:\S_Yamasaki\Projects\Academic\PolysomeFLcDNAseq__A_thaliana\T87-NC-HS\Analysis.FujimakiData.1.4-Takazawa のディレクトリ

2024/08/26  10:49    <DIR>          .
2024/08/26  10:45    <DIR>          ..
2024/08/26  10:49    <DIR>          .ipynb_checkpoints
2021/05/31  09:38                31 _activate_jupyter.bat
2024/08/26  10:49           139,369 step1_data_preparation-1-PR.ipynb
               2 個のファイル             139,400 バイト
               3 個のディレクトリ  3,463,533,973,504 バイトの空き領域


# inputデータ形式の確認

## version 1.4.0 (all version)
- +鎖でも-鎖でもTSSが0b、CPSは1b
- TSSとCPSは同じ出現頻度ならより遠い位置にあるものが採用されている（ペアの場合は、TSSとCPSの遠さの合計が遠いもの、それも同着ならTSSが遠いもの）
- 5'UTRや3'UTRの長さなどは、TSSとCPSのペアとして最も出現頻度が高い位置に基づいて計算されている
- TSSのアノテーションの最低長は1000、DB上のTSSよりも最低でも500上流からアノテーションを行っている
- CPSのアノテーションの最低長は1000、DB上のCPSよりも最低でも500下流までアノテーションを行っている
- SSのずれの補正は最大で5、よりずれが小さいスプライシングパターンを採用している
- 5'UTRと3'UTRは最低でも1塩基は存在するはず
- Coding Geneのみが対象となっているが、ChrCやMも存在する
- スプライシングパターンは一致しなかったけど同じ領域から転写され、同じ領域で転写終結している場合、その遺伝子id.unknownとして定義している
- 同じ領域から転写されたが、ポリA付加部位が、既知の転写産物より、遠い/近い場合、その遺伝子id.extended3/shortened3として定義している
- 同じ領域で転写終結するが、転写開始点が、既知の転写産物より、遠い/近い場合、その遺伝子id.extended5/shortened5として定義している
- version1.3.xでは、.unknownなどの定義に、スプライシングサイトが1つ以上一致しているという条件を加えていたが、その条件をなくしている


In [3]:
file_path = os.path.join("..", "Processing.Fujimaki.1.4", "pychopper_trimmed.mod", "AT22-T.pass3.mod.all.tsv.gz")

data_df, metadata = sylib.fileio.load_df(
    file_path, encoding="utf-8",
    delimiter='\t', comment="#",
    header='infer', names=None, index_col=None,
)
metadata.print_minimum_data(label="Input data", logger=logger)
display(data_df.head(10))

for i, col_name in enumerate(data_df.columns):
    print([i, col_name])

del data_df


2024-08-26 10:49:46,167    DEBUG: Input data |   name = AT22-T.pass3.mod.all.tsv.gz
2024-08-26 10:49:46,168    DEBUG: Input data |    md5 = 2caadcc9d687ffb6789c9ea82db8bce2
2024-08-26 10:49:46,168    DEBUG: Input data | md5_gz = 128d9e8530bcffd1a6aaddfba6ace53d


,Chr,transcripts_name,gene_name,TPM,strand,tss_pos,cps_pos,TSS-CPS_pair,most_used_tts_pos_and_count,most_used_cps_pos_and_count,...,cds_len,3'UTR_len,5'UTR_s_out_count,CDS_s_out_count,3'UTR_s_out_count,start_codon_pos,stop_codon_pos,count,read_cigar,ref_cigar
0,Chr1,AT1G01010.1,AT1G01010,0.063154,+,{3667: 1},{5846: 1},"{'3667,5846': 1}",{3667: 1},{5846: 1},...,1290.0,216.0,0.0,5.0,0.0,3759.0,5630.0,1,246M82N281M209N120M100N390M78N153M112N408M,283M82N281M209N120M100N390M78N153M112N461M
1,Chr1,AT1G01010.shortened5,AT1G01010,0.757851,+,"{4136: 1, 4196: 1, 4503: 1, 4827: 1, 4997: 1, ...","{5817: 1, 5899: 6, 5870: 1, 5831: 1, 5834: 2, ...","{'4136,5817': 1, '4196,5899': 1, '4503,5899': ...",{5553: 2},{5899: 6},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,NaN,NaN
2,Chr1,AT1G01020.1,AT1G01020,1.705165,-,"{8667: 1, 8674: 1, 8675: 1, 8676: 1, 8678: 1, ...","{6882: 2, 6859: 6, 6877: 3, 6862: 1, 6872: 1, ...","{'8667,6882': 1, '8674,6859': 1, '8675,6877': ...",{8719: 4},{6856: 7},...,738.0,56.0,0.0,8.0,0.0,8666.0,6914.0,27,211M87N76M151N67M113N86M112N74M106N46M248N90M9...,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
3,Chr1,AT1G01020.3,AT1G01020,0.063154,-,{8677: 1},{6856: 1},"{'8677,6856': 1}",{8677: 1},{6856: 1},...,711.0,59.0,1.0,6.0,0.0,8442.0,6914.0,1,214M87N76M151N67M113N86M112N74M106N46M248N229M...,282M87N76M151N67M113N86M112N74M106N46M248N229M...
4,Chr1,AT1G01020.5,AT1G01020,1.768319,-,"{8662: 1, 8672: 1, 8673: 1, 8674: 1, 8676: 2, ...","{6877: 4, 6836: 1, 6908: 1, 6790: 1, 6860: 1, ...","{'8662,6877': 1, '8672,6877': 1, '8673,6836': ...",{8680: 4},{6856: 8},...,597.0,59.0,1.0,7.0,0.0,8419.0,6914.0,28,214M87N76M151N67M113N86M112N74M106N46M248N90M9...,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
5,Chr1,AT1G01020.shortened5,AT1G01020,5.431267,-,"{6518: 1, 6873: 1, 6953: 2, 6956: 2, 6965: 1, ...","{5926: 1, 6793: 3, 6856: 9, 6859: 14, 6812: 6,...","{'6518,5926': 1, '6873,6793': 1, '6953,6856': ...",{7227: 13},{6859: 14},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86,NaN,NaN
6,Chr1,AT1G01020.unknown,AT1G01020,7.641666,-,"{8449: 1, 8450: 2, 8463: 2, 8464: 1, 8587: 1, ...","{6801: 1, 6836: 1, 6856: 18, 6884: 1, 6925: 10...","{'8449,6801': 1, '8450,6836': 1, '8450,6856': ...",{8719: 12},{6856: 18},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121,NaN,NaN
7,Chr1,AT1G01040.shortened3,AT1G01040,0.189463,+,"{23130: 1, 23223: 1, 23513: 1}","{25068: 1, 24227: 1, 24091: 1}","{'23130,25068': 1, '23223,24227': 1, '23513,24...",{23130: 1},{25068: 1},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN
8,Chr1,AT1G01040.shortened5,AT1G01040,4.294490,+,"{27096: 1, 27645: 1, 29537: 1, 29550: 1, 29555...","{31184: 4, 31189: 1, 31151: 6, 31176: 15, 3122...","{'27096,31184': 1, '27645,31189': 1, '29537,31...",{31027: 6},{31176: 15},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68,NaN,NaN
9,Chr1,AT1G01050.1,AT1G01050,204.872427,-,"{33045: 3, 33046: 1, 33048: 5, 33049: 9, 33050...","{31197: 149, 31203: 311, 31220: 228, 31147: 22...","{'33045,31197': 1, '33045,31203': 1, '33045,31...",{33113: 1096},{31207: 742},...,639.0,175.0,1.0,7.0,0.0,32670.0,31381.0,3244,218M96N82M90N121M119N66M89N108M86N66M83N29M87N...,255M96N82M90N121M119N66M89N108M86N66M83N29M87N...


[0, 'Chr']
[1, 'transcripts_name']
[2, 'gene_name']
[3, 'TPM']
[4, 'strand']
[5, 'tss_pos']
[6, 'cps_pos']
[7, 'TSS-CPS_pair']
[8, 'most_used_tts_pos_and_count']
[9, 'most_used_cps_pos_and_count']
[10, 'most_numerous_pair_list']
[11, "5'UTR_len"]
[12, 'cds_len']
[13, "3'UTR_len"]
[14, "5'UTR_s_out_count"]
[15, 'CDS_s_out_count']
[16, "3'UTR_s_out_count"]
[17, 'start_codon_pos']
[18, 'stop_codon_pos']
[19, 'count']
[20, 'read_cigar']
[21, 'ref_cigar']


# reference


In [4]:
file_path = os.path.join("..", "ReferenceGenome", "gfd0", "Araport11_GFF3_genes_transposons.201606.std_r_luc.gfd.gz")
gfd0_data, metadata = sylib.fileio.load_json(
    file_path, encoding="utf-8"
)
metadata.print_minimum_data(label="GFD0 data", logger=logger)

file_path = os.path.join("..", "ReferenceGenome", "Arabidopsis_thaliana.Araport11.std_r_luc.fa")
fasta_data, metadata = sylib.fileio.load_fasta(file_path)
metadata.print_minimum_data(label="FASTA data", logger=logger)


2024-08-26 10:49:47,912    DEBUG: GFD0 data |   name = Araport11_GFF3_genes_transposons.201606.std_r_luc.gfd.gz
2024-08-26 10:49:47,913    DEBUG: GFD0 data |    md5 = 8f4870f39f1e7e09623f260b9934cd9e
2024-08-26 10:49:47,913    DEBUG: GFD0 data | md5_gz = 121c0d84d28c1b38316461c5bed74b5f
2024-08-26 10:49:48,522    DEBUG: FASTA data | name = Arabidopsis_thaliana.Araport11.std_r_luc.fa
2024-08-26 10:49:48,522    DEBUG: FASTA data |  md5 = 9883694aa11a04e0868911fd1d40cc1c


# 定数と関数


In [5]:
# DAME_CHROM_SET = {"ChrC", "ChrM", "std_r_luc_expr_cassette"}
DAME_CHROM_SET = {"ChrC", "ChrM"}

UK_LABEL = "unknown"
E5_LABEL = "extended5"
S5_LABEL = "shortened5"
E3_LABEL = "extended3"
S3_LABEL = "shortened3"
UKLABEL_LIST = [UK_LABEL, E5_LABEL, S5_LABEL, E3_LABEL, S3_LABEL]
UKLABEL_SET = set(UKLABEL_LIST)
UKLABEL_IDX_DICT = {name: i for i, name in enumerate(UKLABEL_LIST)}

def main_process(poly_file_path, total_file_path, output_base_file_name):

    p_data_df, metadata = sylib.fileio.load_df(
        poly_file_path, encoding="utf-8",
        delimiter='\t', comment="#",
        header='infer', names=None, index_col=None,
    )
    print(json.dumps(metadata.get_minimum_data_dict(), indent=4))
    # display(data_df)

    t_data_df, metadata = sylib.fileio.load_df(
        total_file_path, encoding="utf-8",
        delimiter='\t', comment="#",
        header='infer', names=None, index_col=None,
    )
    print(json.dumps(metadata.get_minimum_data_dict(), indent=4))
    # display(data_df)

    def get_major_trans_dict(data_df):
        unique_gene_name_list = sorted(set(data_df["gene_name"]))
        # 0 trans_id, 1 idx, 2 n_reads, 3 tpm, 4 sum n_reads, 5 sum tpm, 6 n_reads list, 7 tpm list, 8 n known trans
        # n_reads/tpm list 0 known, 1 unknown, 2  extended5, 3 shortened5, 4 extended3, 5 shortened3
        major_trans_dict = {name: [None, None, 0, 0.0, 0, 0.0, [0]*6, [0.0]*6, 0] for name in unique_gene_name_list}
        gene_idx_dict = {name: [] for name in unique_gene_name_list}
        for i, (trans_name, gene_name, chrom, tpm, n_reads) in enumerate(
            zip(data_df["transcripts_name"], data_df["gene_name"], data_df["Chr"], data_df["TPM"], data_df["count"])
        ):
            if chrom in DAME_CHROM_SET:
                continue
            major_trans_dict[gene_name][4] += n_reads
            major_trans_dict[gene_name][5] += tpm
            uk_idx = UKLABEL_IDX_DICT.get(trans_name.rsplit(".", 1)[-1], None)
            if uk_idx is None:
                major_trans_dict[gene_name][6][0] += n_reads
                major_trans_dict[gene_name][7][0] += tpm
            else:
                major_trans_dict[gene_name][6][uk_idx+1] += n_reads
                major_trans_dict[gene_name][7][uk_idx+1] += tpm
                continue
            major_trans_dict[gene_name][8] += 1
            if n_reads > major_trans_dict[gene_name][2]:
                major_trans_dict[gene_name][0] = trans_name
                major_trans_dict[gene_name][1] = i
                major_trans_dict[gene_name][2] = n_reads
                major_trans_dict[gene_name][3] = tpm
            gene_idx_dict[gene_name].append(i)
        for gene_name in unique_gene_name_list:
            if major_trans_dict[gene_name][0] is None:
                del major_trans_dict[gene_name]
                del gene_idx_dict[gene_name]
        return major_trans_dict, gene_idx_dict
    
    p_major_trans_dict, p_gene_idx_dict = get_major_trans_dict(p_data_df)
    t_major_trans_dict, t_gene_idx_dict = get_major_trans_dict(t_data_df)
    p_trans_id_idx_dict = {trans_id: i for i, trans_id in enumerate(p_data_df["transcripts_name"])}
    t_trans_id_idx_dict = {trans_id: i for i, trans_id in enumerate(t_data_df["transcripts_name"])}
    
    for gene_name, tpm, n_reads in zip(p_data_df["gene_name"], p_data_df["TPM"], p_data_df["count"]):
        if gene_name == "gene:Standard-R-luc":
            p_cx_value = tpm
            p_n_cx_value = n_reads
    for gene_name, tpm, n_reads in zip(t_data_df["gene_name"], t_data_df["TPM"], t_data_df["count"]):
        if gene_name == "gene:Standard-R-luc":
            t_cx_value = tpm
            t_n_cx_value = n_reads
    print("P-correction-value for TPM = "+str(p_cx_value))    
    print("T-correction-value for TPM = "+str(t_cx_value))    
    print("P-correction-value for n reads = "+str(p_n_cx_value))    
    print("T-correction-value for n reads = "+str(t_n_cx_value))    

    print("-"*119)    

    # rTSS系列とrCPS系列には全てのUnknown系列のデータが含まれていない
    # 他のTPM等には、unknownだけが含まれている
    # 使用頻度などのa系列は、knownのTPMで補正されている
    # 使用頻度などのb系列は、known+unknownのTPMで補正されている
    # 使用頻度などのc系列は、known+全unknown系列のTPMで補正されている
    gene_df_dict = {
        "gene_id": [],
        "chrom": [],
        "strand": [],
        "PTPM": [],
        "TPM": [],
        "PR_gene": [],
        "P_n_reads": [],
        "T_n_reads": [],
        "rTrans-id": [],
        "rTrans-usage.a": [],
        "rTrans-usage.b": [],
        "rTrans-usage.c": [],
        "Unknown-ratio.b": [],
        "Unknown-ratio.c": [],
        "5-exteded-ratio.c": [],
        "5-shortened-ratio.c": [],
        "3-exteded-ratio.c": [],
        "3-shortened-ratio.c": [],
        "rTSS-pos0b": [],
        "rTSS-usage.a": [],
        "rCPS-pos1b": [],
        "rCPS-usage.a": [],
        "rTrans-TSS-CPS-id": [],
        "rTrans-TSS-CPS-usage.a": [],
        "rTrans-TSS-CPS-usage.b": [],
        "rTrans-TSS-CPS-usage.c": [],
        "rTrans_coding_region": [],
        "rTrans_ref_cigar_s0b": [],
        "rTrans_ref_cigar": []
    }
    
    progress_bar.set_total_steps(len(t_gene_idx_dict))
    for gene_name in t_gene_idx_dict:
        rtrans_id = t_major_trans_dict[gene_name][0]
        trans_data = gfd0_data["genes"][gene_name]["transcripts"].get(rtrans_id)
        if not trans_data["is_protein_coding"]:
            progress_bar.increase(1)
            continue
        strand = t_data_df.at[t_gene_idx_dict[gene_name][0], "strand"]
        chrom = t_data_df.at[t_gene_idx_dict[gene_name][0], "Chr"]
        try:
            ptpm = p_major_trans_dict[gene_name][7][0]+p_major_trans_dict[gene_name][7][1]
            p_n_reads = p_major_trans_dict[gene_name][6][0]+p_major_trans_dict[gene_name][6][1]
        except KeyError:
            ptpm = 0.0
            p_n_reads = 0
        tpm = t_major_trans_dict[gene_name][7][0]+t_major_trans_dict[gene_name][7][1]
        t_n_reads = t_major_trans_dict[gene_name][6][0]+t_major_trans_dict[gene_name][6][1]
        pr = (ptpm/p_cx_value) / (tpm/t_cx_value)
        gene_df_dict["gene_id"].append(gene_name)
        gene_df_dict["chrom"].append(chrom)
        gene_df_dict["strand"].append(strand)
        gene_df_dict["PTPM"].append(ptpm)
        gene_df_dict["TPM"].append(tpm)
        gene_df_dict["PR_gene"].append(pr)
        gene_df_dict["P_n_reads"].append(p_n_reads)
        gene_df_dict["T_n_reads"].append(t_n_reads)
        gene_df_dict["rTrans-id"].append(t_major_trans_dict[gene_name][0])
        gene_df_dict["rTrans-usage.a"].append(
            t_major_trans_dict[gene_name][3] / t_major_trans_dict[gene_name][7][0]
        )
        gene_df_dict["rTrans-usage.b"].append(
            t_major_trans_dict[gene_name][3] / tpm
        )
        gene_df_dict["rTrans-usage.c"].append(
            t_major_trans_dict[gene_name][3] / t_major_trans_dict[gene_name][5]
        )
        gene_df_dict["Unknown-ratio.b"].append(
            t_major_trans_dict[gene_name][7][1] / tpm
        )
        gene_df_dict["Unknown-ratio.c"].append(
            t_major_trans_dict[gene_name][7][1] / t_major_trans_dict[gene_name][5]
        )
        gene_df_dict["5-exteded-ratio.c"].append(
            t_major_trans_dict[gene_name][7][2] / t_major_trans_dict[gene_name][5]
        )
        gene_df_dict["5-shortened-ratio.c"].append(
            t_major_trans_dict[gene_name][7][3] / t_major_trans_dict[gene_name][5]
        )
        gene_df_dict["3-exteded-ratio.c"].append(
            t_major_trans_dict[gene_name][7][4] / t_major_trans_dict[gene_name][5]
        )
        gene_df_dict["3-shortened-ratio.c"].append(
            t_major_trans_dict[gene_name][7][5] / t_major_trans_dict[gene_name][5]
        )

        tss_dict = {}  # 0-based
        cps_dict = {}  # 1-based
        for i in t_gene_idx_dict[gene_name]:
            trans_name = t_data_df.at[i, "transcripts_name"]
            temp_tss_dict = eval(t_data_df.at[i, "tss_pos"])  # 0-based
            temp_cps_dict = eval(t_data_df.at[i, "cps_pos"])  # 1-based
            for key, value in temp_tss_dict.items():
                try:
                    tss_dict[key] += value
                except KeyError:
                    tss_dict[key] = value
            for key, value in temp_cps_dict.items():
                try:
                    cps_dict[key] += value
                except KeyError:
                    cps_dict[key] = value

        rtss_pos, rtss_n_reads = None, 0
        for pos in sorted(tss_dict.keys()):
            if tss_dict[pos] > rtss_n_reads:
                rtss_pos, rtss_n_reads = pos, tss_dict[pos]
            elif strand == "-" and tss_dict[pos] == rtss_n_reads:
                # マイナス鎖なら同じスコアでも更新することによって遠いTSSを優先するようにする
                rtss_pos = pos
        rcps_pos, rcps_n_reads = None, 0
        for pos in sorted(cps_dict.keys()):
            if cps_dict[pos] > rcps_n_reads:
                rcps_pos, rcps_n_reads = pos, cps_dict[pos]
            elif strand == "+" and cps_dict[pos] == rcps_n_reads:
                # プラス鎖なら同じスコアでも更新することによって遠いCPSを優先するようにする
                rcps_pos = pos

        rtrans_idx = t_major_trans_dict[gene_name][1]
        r_both_ends_dict = eval(t_data_df.at[rtrans_idx, "most_numerous_pair_list"])
        rtss_both_ends_id = (
            t_major_trans_dict[gene_name][0]+","+list(r_both_ends_dict.keys())[0]
        )
        rtss_both_ends_tpm = (
             t_data_df.at[rtrans_idx, "TPM"]
            * (list(r_both_ends_dict.values())[0]/t_data_df.at[rtrans_idx, "count"])
        )

        gene_df_dict["rTSS-pos0b"].append(rtss_pos)
        gene_df_dict["rTSS-usage.a"].append(rtss_n_reads / t_major_trans_dict[gene_name][6][0])
        gene_df_dict["rCPS-pos1b"].append(rcps_pos)
        gene_df_dict["rCPS-usage.a"].append(rcps_n_reads / t_major_trans_dict[gene_name][6][0])
        gene_df_dict["rTrans-TSS-CPS-id"].append(rtss_both_ends_id)
        gene_df_dict["rTrans-TSS-CPS-usage.a"].append(rtss_both_ends_tpm / t_major_trans_dict[gene_name][7][0])
        gene_df_dict["rTrans-TSS-CPS-usage.b"].append(rtss_both_ends_tpm / tpm)
        gene_df_dict["rTrans-TSS-CPS-usage.c"].append(rtss_both_ends_tpm / t_major_trans_dict[gene_name][5])
        gene_df_dict["rTrans_coding_region"].append(trans_data["coding_region"])
        gene_df_dict["rTrans_ref_cigar_s0b"].append(trans_data["region"][0])
        gene_df_dict["rTrans_ref_cigar"].append(trans_data["cigar"])
        progress_bar.increase(1)
    progress_bar.close()

    gene_df = pd.DataFrame(gene_df_dict)
    print("n_gene_data = %s" % len(gene_df))

    output_path = output_base_file_name + ".gene_data.tsv.gz"
    metadata = sylib.fileio.write_df(
        gene_df, output_path, header_comment_list=None,
        footer_comment_list=None, encoding="utf-8",
        sep="\t", header=True, index=False, line_terminator="\n",
        gzip_mtime=0
    )
    print(json.dumps(metadata.get_minimum_data_dict(), indent=4))

    display(gene_df)
    for i, col_name in enumerate(gene_df.columns):
        print([i, col_name])

    print("-"*119)
    
    def get_pos_data_dict(p_pos_data_dict, t_pos_data_dict):
        pos_data_dict = {}
        for pos, n_reads in t_pos_data_dict.items():
            pos_data_dict[pos] = [0, n_reads, None]
        for pos, n_reads in p_pos_data_dict.items():
            try:
                pos_data_dict[pos][0] = n_reads
            except KeyError:
                pos_data_dict[pos] = [n_reads, 0, None]
        pos_data_dict = sylib.utils.sort_dict_by_key(pos_data_dict)
        for pos in pos_data_dict:
            if pos_data_dict[pos][1] == 0:
                pos_data_dict[pos][2] = None
            else:
                pos_data_dict[pos][2] = (pos_data_dict[pos][0]/p_n_cx_value) / (pos_data_dict[pos][1]/t_n_cx_value)
        return pos_data_dict
    
    def calc_end_coord_pvalue(tss_data_dict, cps_data_dict, both_ends_data, n_reads):
        if n_reads < 13:
            return 0.0, 1.0
        other_exp, other_obs = 0.0, 0
        exp_list, obs_list = [], []
        for tss_pos, tss_n_reads in tss_data_dict.items():
            for cps_pos, cps_n_reads in cps_data_dict.items():
                exp_n_reads = n_reads * (tss_n_reads/n_reads) * (cps_n_reads/n_reads)
                obs_n_reads = both_ends_data.get("%s,%s" % (tss_pos, cps_pos), 0)
                if min(exp_n_reads, obs_n_reads) < 5:
                    other_exp += exp_n_reads
                    other_obs += obs_n_reads
                else:
                    exp_list.append(exp_n_reads)
                    obs_list.append(obs_n_reads)
        if len(exp_list) == 0:
            return 0.0, 1.0
        obs_list.append(other_obs)
        exp_list.append(other_exp)
        _, p_value = chisquare(obs_list, f_exp=exp_list)
        chisq_stats, _ = chisquare(
            [v/sum(obs_list) for v in obs_list], f_exp=[v/sum(exp_list) for v in exp_list]
        )
        return chisq_stats, p_value

    trans_df_dict = {
        "trans_id": [],
        "gene_id": [],
        "chrom": [],
        "strand": [],
        "PTPM": [],
        "TPM": [],
        "PR_trans": [],
        "P_n_reads": [],
        "T_n_reads": [],
        "trans-usage.a": [],
        "trans-usage.b": [],
        "trans-usage.c": [],
        "rTSS-pos0b": [],
        "rTSS-usage": [],
        "rCPS-pos1b": [],
        "rCPS-usage": [],
        "rTSS-CPS-pos": [],
        "rTSS-CPS-usage": [],
        "TSS-data": [],
        "CPS-data": [],
        "TSS-CPS-data": [],
        "P-PairDep-pvalue": [],
        "P-PairDep-RelStats": [],
        "T-PairDep-pvalue": [],
        "T-PairDep-RelStats": [],
        "coding_region": [],
        "ref_cigar_s0b": [],
        "ref_cigar": [],
    }
    n_noncoding_data = 0
    n_dame_chrom_data = 0
    n_uk_trans_data = 0
    progress_bar.set_total_steps(len(t_data_df))
    for i in range(len(t_data_df)):
        gene_name = t_data_df.at[i, "gene_name"]
        trans_name = t_data_df.at[i, "transcripts_name"]
        chromosome = t_data_df.at[i, "Chr"]
        trans_data = gfd0_data["genes"][gene_name]["transcripts"].get(trans_name)
        if chromosome in DAME_CHROM_SET:
            n_dame_chrom_data += 1
            progress_bar.increase(1)
            continue
        if trans_name.rsplit(".", 1)[-1] in UKLABEL_SET:
            n_uk_trans_data += 1
            progress_bar.increase(1)
            continue
        if not trans_data["is_protein_coding"]:
            n_noncoding_data += 1
            progress_bar.increase(1)
            continue

        strand = t_data_df.at[i, "strand"]
        try:
            p_idx = p_trans_id_idx_dict[trans_name]
            ptpm = p_data_df.at[p_idx, "TPM"]
            p_n_reads = p_data_df.at[p_idx, "count"]
            p_tss_data = eval(p_data_df.at[p_idx, "tss_pos"])
            p_cps_data = eval(p_data_df.at[p_idx, "cps_pos"])
            p_both_ends_data = eval(p_data_df.at[p_idx, "TSS-CPS_pair"])
        except KeyError:
            ptpm = 0.0
            p_n_reads = 0
            p_tss_data = {}
            p_cps_data = {}
            p_both_ends_data = {}
        tpm_gene = t_major_trans_dict[gene_name][7][0]+t_major_trans_dict[gene_name][7][1]
        tpm = t_data_df.at[i, "TPM"]
        t_n_reads = t_data_df.at[i, "count"]
        t_tss_data = eval(t_data_df.at[i, "tss_pos"])
        t_cps_data = eval(t_data_df.at[i, "cps_pos"])
        t_both_ends_data = eval(t_data_df.at[i, "TSS-CPS_pair"])
        pr = (ptpm/p_cx_value) / (tpm/t_cx_value)
        coding_region = trans_data["coding_region"]

        trans_df_dict["trans_id"].append(trans_name)
        trans_df_dict["gene_id"].append(gene_name)
        trans_df_dict["chrom"].append(chromosome)
        trans_df_dict["strand"].append(strand)
        trans_df_dict["PTPM"].append(ptpm)
        trans_df_dict["TPM"].append(tpm)
        trans_df_dict["PR_trans"].append(pr)
        trans_df_dict["P_n_reads"].append(p_n_reads)
        trans_df_dict["T_n_reads"].append(t_n_reads)
        trans_df_dict["trans-usage.a"].append(tpm / t_major_trans_dict[gene_name][7][0])
        trans_df_dict["trans-usage.b"].append(tpm / tpm_gene)
        trans_df_dict["trans-usage.c"].append(tpm / t_major_trans_dict[gene_name][5])
        trans_df_dict["rTSS-pos0b"].append(list(eval(t_data_df.at[i, "most_used_tts_pos_and_count"]).keys())[0])
        trans_df_dict["rTSS-usage"].append(list(eval(t_data_df.at[i, "most_used_tts_pos_and_count"]).values())[0]/t_n_reads)
        trans_df_dict["rCPS-pos1b"].append(list(eval(t_data_df.at[i, "most_used_cps_pos_and_count"]).keys())[0])
        trans_df_dict["rCPS-usage"].append(list(eval(t_data_df.at[i, "most_used_cps_pos_and_count"]).values())[0]/t_n_reads)
        rtss_cps_text = list(eval(t_data_df.at[i, "most_numerous_pair_list"]).keys())[0]
        tss0b, cps1b = rtss_cps_text.split(",")
        tss0b, cps1b = int(tss0b), int(cps1b)
        trans_df_dict["rTSS-CPS-pos"].append([tss0b, cps1b])
        trans_df_dict["rTSS-CPS-usage"].append(list(eval(t_data_df.at[i, "most_numerous_pair_list"]).values())[0]/t_n_reads)
        trans_df_dict["TSS-data"].append(get_pos_data_dict(p_tss_data, t_tss_data))
        trans_df_dict["CPS-data"].append(get_pos_data_dict(p_cps_data, t_cps_data))
        trans_df_dict["TSS-CPS-data"].append(get_pos_data_dict(p_both_ends_data, t_both_ends_data))
        rel_stats, p_value = calc_end_coord_pvalue(p_tss_data, p_cps_data, p_both_ends_data, p_n_reads)
        # pair dependency
        trans_df_dict["P-PairDep-pvalue"].append(p_value)
        trans_df_dict["P-PairDep-RelStats"].append(rel_stats)
        rel_stats, p_value = calc_end_coord_pvalue(t_tss_data, t_cps_data, t_both_ends_data, t_n_reads)
        trans_df_dict["T-PairDep-pvalue"].append(p_value)
        trans_df_dict["T-PairDep-RelStats"].append(rel_stats)
        trans_df_dict["coding_region"].append(coding_region)
        trans_df_dict["ref_cigar_s0b"].append(trans_data["region"][0])
        trans_df_dict["ref_cigar"].append(trans_data["cigar"])
        progress_bar.increase(1)
    progress_bar.close()

    # output
    print("p_n_data = %s" % len(p_data_df))
    print("t_n_data = %s" % len(t_data_df))
    print("t_n_dame_chrom_data = %s" % n_dame_chrom_data)
    print("t_n_uk_trans_data = %s" % n_uk_trans_data)
    print("t_n_noncoding_data = %s" % n_noncoding_data)

    trans_df = pd.DataFrame(trans_df_dict)
    print("n_data = %s" % len(trans_df))

    output_path = output_base_file_name + ".trans_data.tsv.gz"
    metadata = sylib.fileio.write_df(
        trans_df, output_path, header_comment_list=None,
        footer_comment_list=None, encoding="utf-8",
        sep="\t", header=True, index=False, line_terminator="\n",
        gzip_mtime=0
    )
    print(json.dumps(metadata.get_minimum_data_dict(), indent=4))

    display(trans_df)
    for i, col_name in enumerate(trans_df.columns):
        print([i, col_name])


# 実行

## NC


In [6]:
poly_file_path = os.path.join("..", "Processing.Fujimaki.1.4", "pychopper_trimmed.mod", "AT22-P.pass3.mod.all.tsv.gz")
total_file_path = os.path.join("..", "Processing.Fujimaki.1.4", "pychopper_trimmed.mod", "AT22-T.pass3.mod.all.tsv.gz")
output_base_file_name = "AT22.PR"

######################################################################

main_process(poly_file_path, total_file_path, output_base_file_name)


{
    "name": "AT22-P.pass3.mod.all.tsv.gz",
    "md5": "530f282266b2ac3794c6faf3a51a0b72",
    "md5_gz": "4a0de35480b3983690fa856b8058f5f8"
}
{
    "name": "AT22-T.pass3.mod.all.tsv.gz",
    "md5": "2caadcc9d687ffb6789c9ea82db8bce2",
    "md5_gz": "128d9e8530bcffd1a6aaddfba6ace53d"
}
P-correction-value for TPM = 2231.6406208751355
T-correction-value for TPM = 2272.100897270498
P-correction-value for n reads = 21979
T-correction-value for n reads = 35977
-----------------------------------------------------------------------------------------------------------------------


100%|██████████████████████████████████████████████████████████████████████████| 16556/16556 [00:01<00:00, 9031.15it/s]
C:\Users\Yamasakibioinfo\AppData\Local\Temp\ipykernel_11484\188375359.py:237: FutureWarning: `line_terminator` will be removed in future release. Please use `lineterminator`.
  metadata = sylib.fileio.write_df(


n_gene_data = 16556
{
    "name": "AT22.PR.gene_data.tsv.gz",
    "md5": "d6483936327cd270d3fb6758aec8a7d0",
    "md5_gz": "fc8a351b6e1bc49b73a92bf2ebb49c86"
}


,gene_id,chrom,strand,PTPM,TPM,PR_gene,P_n_reads,T_n_reads,rTrans-id,rTrans-usage.a,...,rTSS-usage.a,rCPS-pos1b,rCPS-usage.a,rTrans-TSS-CPS-id,rTrans-TSS-CPS-usage.a,rTrans-TSS-CPS-usage.b,rTrans-TSS-CPS-usage.c,rTrans_coding_region,rTrans_ref_cigar_s0b,rTrans_ref_cigar
0,AT1G01010,Chr1,+,0.304605,0.063154,4.910642,3,1,AT1G01010.1,1.000000,...,1.000000,5846,1.000000,"AT1G01010.1,3667,5846",1.000000,1.000000,0.076923,"[3759, 5630]",3630,283M82N281M209N120M100N390M78N153M112N461M
1,AT1G01020,Chr1,-,11.879610,11.178304,1.082006,117,177,AT1G01020.5,0.500000,...,0.125000,6856,0.285714,"AT1G01020.5,8680,6856",0.053571,0.016949,0.011407,"[6914, 8419]",6787,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
2,AT1G01050,Chr1,-,265.006689,223.818706,1.205491,2610,3544,AT1G01050.1,1.000000,...,0.337855,31207,0.228730,"AT1G01050.1,33113,31207",0.077374,0.070824,0.054851,"[31381, 32670]",31169,255M96N82M90N121M119N66M89N108M86N66M83N29M87N...
3,AT1G01060,Chr1,-,0.000000,0.126309,0.000000,0,2,AT1G01060.2,0.500000,...,0.500000,33745,0.500000,"AT1G01060.2,37398,33761",0.500000,0.500000,0.090909,"[33991, 37061]",33665,662M73N1074M92N81M82N234M660N62M124N112M101N18...
4,AT1G01080,Chr1,-,4.569081,3.915564,1.188059,45,62,AT1G01080.1,0.807018,...,0.368421,45352,0.122807,"AT1G01080.1,46825,45352",0.052632,0.048387,0.008000,"[45502, 46789]",44969,590M86N309M89N102M230N684M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16551,AT5G67610,Chr5,-,0.710746,0.631543,1.145816,7,10,AT5G67610.1,1.000000,...,0.500000,26960966,0.500000,"AT5G67610.1,26963640,26960966",0.500000,0.100000,0.010870,"[26961223, 26963614]",26960903,586M81N149M121N107M77N162M127N93M96N60M85N146M...
16552,AT5G67620,Chr5,-,0.304605,0.315771,0.982128,3,5,AT5G67620.1,1.000000,...,0.400000,26964773,0.400000,"AT5G67620.1,26965995,26964773",0.200000,0.200000,0.142857,"[26964890, 26965720]",26964550,454M193N182M88N543M
16553,AT5G67630,Chr5,-,11.778075,8.715288,1.375929,116,138,AT5G67630.1,1.000000,...,0.266667,26967381,0.303704,"AT5G67630.1,26969383,26967381",0.088889,0.086957,0.017266,"[26967534, 26969306]",26967377,818M362N843M
16554,AT5G67640,Chr5,-,6.904389,8.273208,0.849679,68,131,AT5G67640.1,1.000000,...,0.160000,26969475,0.192000,"AT5G67640.1,26970621,26969475",0.040000,0.038168,0.033333,"[26969545, 26970548]",26969515,155M244N327M70N49M83N225M


[0, 'gene_id']
[1, 'chrom']
[2, 'strand']
[3, 'PTPM']
[4, 'TPM']
[5, 'PR_gene']
[6, 'P_n_reads']
[7, 'T_n_reads']
[8, 'rTrans-id']
[9, 'rTrans-usage.a']
[10, 'rTrans-usage.b']
[11, 'rTrans-usage.c']
[12, 'Unknown-ratio.b']
[13, 'Unknown-ratio.c']
[14, '5-exteded-ratio.c']
[15, '5-shortened-ratio.c']
[16, '3-exteded-ratio.c']
[17, '3-shortened-ratio.c']
[18, 'rTSS-pos0b']
[19, 'rTSS-usage.a']
[20, 'rCPS-pos1b']
[21, 'rCPS-usage.a']
[22, 'rTrans-TSS-CPS-id']
[23, 'rTrans-TSS-CPS-usage.a']
[24, 'rTrans-TSS-CPS-usage.b']
[25, 'rTrans-TSS-CPS-usage.c']
[26, 'rTrans_coding_region']
[27, 'rTrans_ref_cigar_s0b']
[28, 'rTrans_ref_cigar']
-----------------------------------------------------------------------------------------------------------------------


100%|██████████████████████████████████████████████████████████████████████████| 75127/75127 [00:29<00:00, 2556.91it/s]
C:\Users\Yamasakibioinfo\AppData\Local\Temp\ipykernel_11484\188375359.py:416: FutureWarning: `line_terminator` will be removed in future release. Please use `lineterminator`.
  metadata = sylib.fileio.write_df(


p_n_data = 69731
t_n_data = 75127
t_n_dame_chrom_data = 669
t_n_uk_trans_data = 51946
t_n_noncoding_data = 0
n_data = 22512
{
    "name": "AT22.PR.trans_data.tsv.gz",
    "md5": "6fcb6b139c7e83fb42a35e72016cfe5c",
    "md5_gz": "5644219c1183afc2e6720dd8541a35b0"
}


,trans_id,gene_id,chrom,strand,PTPM,TPM,PR_trans,P_n_reads,T_n_reads,trans-usage.a,...,TSS-data,CPS-data,TSS-CPS-data,P-PairDep-pvalue,P-PairDep-RelStats,T-PairDep-pvalue,T-PairDep-RelStats,coding_region,ref_cigar_s0b,ref_cigar
0,AT1G01010.1,AT1G01010,Chr1,+,0.203070,0.063154,3.273761,2,1,1.000000,...,"{3638: [1, 0, None], 3667: [1, 1, 1.6368806588...","{5846: [0, 1, 0.0], 5878: [1, 0, None], 5896: ...","{'3638,5878': [1, 0, None], '3667,5846': [0, 1...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[3759, 5630]",3630,283M82N281M209N120M100N390M78N153M112N461M
1,AT1G01020.1,AT1G01020,Chr1,-,3.147589,1.705165,1.879381,31,27,0.482143,...,"{8667: [0, 1, 0.0], 8669: [1, 0, None], 8670: ...","{6342: [1, 0, None], 6790: [1, 0, None], 6795:...","{'8667,6882': [0, 1, 0.0], '8669,6342': [1, 0,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[6914, 8666]",6787,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
2,AT1G01020.3,AT1G01020,Chr1,-,0.000000,0.063154,0.000000,0,1,0.017857,...,"{8677: [0, 1, 0.0]}","{6856: [0, 1, 0.0]}","{'8677,6856': [0, 1, 0.0]}",1.000000e+00,0.000000,1.000000e+00,0.000000,"[6914, 8442]",6787,282M87N76M151N67M113N86M112N74M106N46M248N229M...
3,AT1G01020.5,AT1G01020,Chr1,-,1.319957,1.768319,0.759980,13,28,0.500000,...,"{8662: [0, 1, 0.0], 8671: [1, 0, None], 8672: ...","{6790: [0, 1, 0.0], 6828: [1, 0, None], 6833: ...","{'8662,6877': [0, 1, 0.0], '8671,6859': [1, 0,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[6914, 8419]",6787,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
4,AT1G01050.1,AT1G01050,Chr1,-,244.293523,204.872427,1.214037,2406,3244,1.000000,...,"{33045: [1, 3, 0.5456268862702276], 33046: [1,...","{31053: [0, 1, 0.0], 31061: [2, 3, 1.091253772...","{'33045,31181': [1, 0, None], '33045,31197': [...",9.179746e-01,0.019254,9.799727e-01,0.017847,"[31381, 32670]",31169,255M96N82M90N121M119N66M89N108M86N66M83N29M87N...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22507,AT5G67610.1,AT5G67610,Chr5,-,0.710746,0.126309,5.729082,7,2,1.000000,...,"{26963614: [1, 0, None], 26963640: [1, 1, 1.63...","{26960965: [1, 0, None], 26960966: [0, 1, 0.0]...","{'26963614,26961018': [1, 0, None], '26963640,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26961223, 26963614]",26960903,586M81N149M121N107M77N162M127N93M96N60M85N146M...
22508,AT5G67620.1,AT5G67620,Chr5,-,0.304605,0.315771,0.982128,3,5,1.000000,...,"{26965773: [0, 1, 0.0], 26965774: [3, 0, None]...","{26964739: [0, 1, 0.0], 26964773: [3, 2, 2.455...","{'26965773,26964773': [0, 1, 0.0], '26965774,2...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26964890, 26965720]",26964550,454M193N182M88N543M
22509,AT5G67630.1,AT5G67630,Chr5,-,11.676540,8.525825,1.394380,115,135,1.000000,...,"{26969313: [1, 0, None], 26969317: [1, 1, 1.63...","{26967369: [0, 1, 0.0], 26967377: [1, 0, None]...","{'26969313,26967421': [1, 0, None], '26969317,...",3.801085e-01,0.006699,7.766911e-01,0.003744,"[26967534, 26969306]",26967377,818M362N843M
22510,AT5G67640.1,AT5G67640,Chr5,-,6.599783,7.894283,0.851178,65,125,1.000000,...,"{26970548: [1, 3, 0.5456268862702276], 2697054...","{26969348: [0, 1, 0.0], 26969351: [0, 1, 0.0],...","{'26970548,26969351': [0, 1, 0.0], '26970548,2...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26969545, 26970548]",26969515,155M244N327M70N49M83N225M


[0, 'trans_id']
[1, 'gene_id']
[2, 'chrom']
[3, 'strand']
[4, 'PTPM']
[5, 'TPM']
[6, 'PR_trans']
[7, 'P_n_reads']
[8, 'T_n_reads']
[9, 'trans-usage.a']
[10, 'trans-usage.b']
[11, 'trans-usage.c']
[12, 'rTSS-pos0b']
[13, 'rTSS-usage']
[14, 'rCPS-pos1b']
[15, 'rCPS-usage']
[16, 'rTSS-CPS-pos']
[17, 'rTSS-CPS-usage']
[18, 'TSS-data']
[19, 'CPS-data']
[20, 'TSS-CPS-data']
[21, 'P-PairDep-pvalue']
[22, 'P-PairDep-RelStats']
[23, 'T-PairDep-pvalue']
[24, 'T-PairDep-RelStats']
[25, 'coding_region']
[26, 'ref_cigar_s0b']
[27, 'ref_cigar']


## HS


In [7]:
poly_file_path = os.path.join("..", "Processing.Fujimaki.1.4", "pychopper_trimmed.mod", "AT35-P.pass3.mod.all.tsv.gz")
total_file_path = os.path.join("..", "Processing.Fujimaki.1.4", "pychopper_trimmed.mod", "AT35-T.pass3.mod.all.tsv.gz")
output_base_file_name = "AT35.PR"

######################################################################

main_process(poly_file_path, total_file_path, output_base_file_name)


{
    "name": "AT35-P.pass3.mod.all.tsv.gz",
    "md5": "617ad0ebc2699fd62c138b93fb42c5d1",
    "md5_gz": "f6fe183e609b547cfc79755168f57e24"
}
{
    "name": "AT35-T.pass3.mod.all.tsv.gz",
    "md5": "c517e1607daa59fd81cb98e6fb920bb2",
    "md5_gz": "192c69343348a85279c5927ee3ff992c"
}
P-correction-value for TPM = 2449.3275220596647
T-correction-value for TPM = 1998.3161090585756
P-correction-value for n reads = 24389
T-correction-value for n reads = 21831
-----------------------------------------------------------------------------------------------------------------------


100%|█████████████████████████████████████████████████████████████████████████| 15053/15053 [00:01<00:00, 10278.16it/s]
C:\Users\Yamasakibioinfo\AppData\Local\Temp\ipykernel_11484\188375359.py:237: FutureWarning: `line_terminator` will be removed in future release. Please use `lineterminator`.
  metadata = sylib.fileio.write_df(


n_gene_data = 15053
{
    "name": "AT35.PR.gene_data.tsv.gz",
    "md5": "bb654d8182b2c21f3c002b558b282ec1",
    "md5_gz": "93568812e7ec618026fa2ad174caa06b"
}


,gene_id,chrom,strand,PTPM,TPM,PR_gene,P_n_reads,T_n_reads,rTrans-id,rTrans-usage.a,...,rTSS-usage.a,rCPS-pos1b,rCPS-usage.a,rTrans-TSS-CPS-id,rTrans-TSS-CPS-usage.a,rTrans-TSS-CPS-usage.b,rTrans-TSS-CPS-usage.c,rTrans_coding_region,rTrans_ref_cigar_s0b,rTrans_ref_cigar
0,AT1G01010,Chr1,+,0.702993,0.091536,6.265817,7,1,AT1G01010.1,1.000000,...,1.000000,5846,1.000000,"AT1G01010.1,3668,5846",1.000000,1.000000,0.250000,"[3759, 5630]",3630,283M82N281M209N120M100N390M78N153M112N461M
1,AT1G01020,Chr1,-,10.846175,17.849464,0.495757,108,195,AT1G01020.1,0.522727,...,0.227273,6856,0.181818,"AT1G01020.1,8719,6856",0.045455,0.010256,0.006689,"[6914, 8666]",6787,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
2,AT1G01050,Chr1,-,152.951159,146.274066,0.853106,1523,1598,AT1G01050.1,0.999308,...,0.308651,31207,0.255363,"AT1G01050.1,33113,31207",0.072664,0.065707,0.053191,"[31381, 32670]",31169,255M96N82M90N121M119N66M89N108M86N66M83N29M87N...
3,AT1G01060,Chr1,-,0.000000,0.091536,0.000000,0,1,AT1G01060.7,1.000000,...,1.000000,33733,1.000000,"AT1G01060.7,37778,33733",1.000000,1.000000,0.040000,"[33991, 37061]",33661,666M73N1074M92N81M82N234M660N62M124N112M101N37...
4,AT1G01080,Chr1,-,2.912399,2.196857,1.081599,29,24,AT1G01080.1,0.947368,...,0.315789,45406,0.157895,"AT1G01080.1,46825,45406",0.157895,0.125000,0.023622,"[45502, 46789]",44969,590M86N309M89N102M230N684M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15048,AT5G67610,Chr5,-,0.502138,0.274607,1.491861,5,3,AT5G67610.1,1.000000,...,1.000000,26960918,1.000000,"AT5G67610.1,26963665,26960918",1.000000,0.333333,0.018519,"[26961223, 26963614]",26960903,586M81N149M121N107M77N162M127N93M96N60M85N146M...
15049,AT5G67620,Chr5,-,0.000000,0.274607,0.000000,0,3,AT5G67620.1,1.000000,...,0.333333,26964760,0.666667,"AT5G67620.1,26965995,26964775",0.333333,0.333333,0.333333,"[26964890, 26965720]",26964550,454M193N182M88N543M
15050,AT5G67630,Chr5,-,7.431639,4.759857,1.273820,74,52,AT5G67630.1,1.000000,...,0.200000,26967381,0.360000,"AT5G67630.1,26969383,26967381",0.080000,0.076923,0.016194,"[26967534, 26969306]",26967377,818M362N843M
15051,AT5G67640,Chr5,-,4.318385,6.956714,0.506448,43,76,AT5G67640.1,1.000000,...,0.126761,26969435,0.197183,"AT5G67640.1,26970621,26969435",0.028169,0.026316,0.020833,"[26969545, 26970548]",26969515,155M244N327M70N49M83N225M


[0, 'gene_id']
[1, 'chrom']
[2, 'strand']
[3, 'PTPM']
[4, 'TPM']
[5, 'PR_gene']
[6, 'P_n_reads']
[7, 'T_n_reads']
[8, 'rTrans-id']
[9, 'rTrans-usage.a']
[10, 'rTrans-usage.b']
[11, 'rTrans-usage.c']
[12, 'Unknown-ratio.b']
[13, 'Unknown-ratio.c']
[14, '5-exteded-ratio.c']
[15, '5-shortened-ratio.c']
[16, '3-exteded-ratio.c']
[17, '3-shortened-ratio.c']
[18, 'rTSS-pos0b']
[19, 'rTSS-usage.a']
[20, 'rCPS-pos1b']
[21, 'rCPS-usage.a']
[22, 'rTrans-TSS-CPS-id']
[23, 'rTrans-TSS-CPS-usage.a']
[24, 'rTrans-TSS-CPS-usage.b']
[25, 'rTrans-TSS-CPS-usage.c']
[26, 'rTrans_coding_region']
[27, 'rTrans_ref_cigar_s0b']
[28, 'rTrans_ref_cigar']
-----------------------------------------------------------------------------------------------------------------------


100%|██████████████████████████████████████████████████████████████████████████| 67678/67678 [00:22<00:00, 2993.40it/s]
C:\Users\Yamasakibioinfo\AppData\Local\Temp\ipykernel_11484\188375359.py:416: FutureWarning: `line_terminator` will be removed in future release. Please use `lineterminator`.
  metadata = sylib.fileio.write_df(


p_n_data = 68190
t_n_data = 67678
t_n_dame_chrom_data = 735
t_n_uk_trans_data = 47350
t_n_noncoding_data = 0
n_data = 19593
{
    "name": "AT35.PR.trans_data.tsv.gz",
    "md5": "4973645d049d85cfeda2adf05abd37a4",
    "md5_gz": "560080f42ae6921752fda29ccc0f8132"
}


,trans_id,gene_id,chrom,strand,PTPM,TPM,PR_trans,P_n_reads,T_n_reads,trans-usage.a,...,TSS-data,CPS-data,TSS-CPS-data,P-PairDep-pvalue,P-PairDep-RelStats,T-PairDep-pvalue,T-PairDep-RelStats,coding_region,ref_cigar_s0b,ref_cigar
0,AT1G01010.1,AT1G01010,Chr1,+,0.502138,0.091536,4.475583,5,1,1.000000,...,"{3626: [2, 0, None], 3628: [1, 0, None], 3630:...","{5844: [1, 0, None], 5846: [0, 1, 0.0], 5870: ...","{'3626,5880': [2, 0, None], '3628,5870': [1, 0...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[3759, 5630]",3630,283M82N281M209N120M100N390M78N153M112N461M
1,AT1G01020.1,AT1G01020,Chr1,-,2.008551,2.105321,0.778362,20,23,0.522727,...,"{8671: [0, 1, 0.0], 8672: [2, 3, 0.59674443396...","{6782: [1, 0, None], 6790: [3, 0, None], 6800:...","{'8671,6800': [0, 1, 0.0], '8672,6800': [0, 1,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[6914, 8666]",6787,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
2,AT1G01020.3,AT1G01020,Chr1,-,0.100428,0.091536,0.895117,1,1,0.022727,...,"{8690: [1, 0, None], 8692: [0, 1, 0.0]}","{6828: [1, 0, None], 6856: [0, 1, 0.0]}","{'8690,6828': [1, 0, None], '8692,6856': [0, 1...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[6914, 8442]",6787,282M87N76M151N67M113N86M112N74M106N46M248N229M...
3,AT1G01020.5,AT1G01020,Chr1,-,0.702993,1.830714,0.313291,7,20,0.454545,...,"{8669: [0, 1, 0.0], 8675: [1, 0, None], 8676: ...","{6775: [0, 1, 0.0], 6790: [0, 2, 0.0], 6791: [...","{'8669,6856': [0, 1, 0.0], '8675,6859': [1, 0,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[6914, 8419]",6787,282M87N76M151N67M113N86M112N74M106N46M248N90M9...
4,AT1G01050.1,AT1G01050,Chr1,-,142.205411,132.177567,0.877760,1416,1444,0.999308,...,"{33045: [0, 1, 0.0], 33046: [1, 3, 0.298372216...","{31061: [0, 3, 0.0], 31075: [0, 1, 0.0], 31113...","{'33045,31181': [0, 1, 0.0], '33046,31186': [0...",7.760653e-01,0.019439,3.660240e-01,0.027956,"[31381, 32670]",31169,255M96N82M90N121M119N66M89N108M86N66M83N29M87N...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19588,AT5G67610.1,AT5G67610,Chr5,-,0.200855,0.091536,1.790233,2,1,1.000000,...,"{26963614: [1, 0, None], 26963665: [0, 1, 0.0]...","{26960918: [0, 1, 0.0], 26960988: [1, 0, None]...","{'26963614,26960988': [1, 0, None], '26963665,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26961223, 26963614]",26960903,586M81N149M121N107M77N162M127N93M96N60M85N146M...
19589,AT5G67620.1,AT5G67620,Chr5,-,0.000000,0.274607,0.000000,0,3,1.000000,...,"{26965865: [0, 1, 0.0], 26965906: [0, 1, 0.0],...","{26964760: [0, 2, 0.0], 26964775: [0, 1, 0.0]}","{'26965865,26964760': [0, 1, 0.0], '26965906,2...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26964890, 26965720]",26964550,454M193N182M88N543M
19590,AT5G67630.1,AT5G67630,Chr5,-,7.431639,4.576786,1.324773,74,50,1.000000,...,"{26969313: [1, 0, None], 26969318: [1, 0, None...","{26967350: [0, 1, 0.0], 26967368: [0, 1, 0.0],...","{'26969313,26967381': [1, 0, None], '26969318,...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26967534, 26969306]",26967377,818M362N843M
19591,AT5G67640.1,AT5G67640,Chr5,-,3.816247,6.499035,0.479077,38,71,1.000000,...,"{26970549: [0, 1, 0.0], 26970551: [0, 1, 0.0],...","{26969381: [0, 1, 0.0], 26969398: [0, 6, 0.0],...","{'26970549,26969519': [0, 1, 0.0], '26970551,2...",1.000000e+00,0.000000,1.000000e+00,0.000000,"[26969545, 26970548]",26969515,155M244N327M70N49M83N225M


[0, 'trans_id']
[1, 'gene_id']
[2, 'chrom']
[3, 'strand']
[4, 'PTPM']
[5, 'TPM']
[6, 'PR_trans']
[7, 'P_n_reads']
[8, 'T_n_reads']
[9, 'trans-usage.a']
[10, 'trans-usage.b']
[11, 'trans-usage.c']
[12, 'rTSS-pos0b']
[13, 'rTSS-usage']
[14, 'rCPS-pos1b']
[15, 'rCPS-usage']
[16, 'rTSS-CPS-pos']
[17, 'rTSS-CPS-usage']
[18, 'TSS-data']
[19, 'CPS-data']
[20, 'TSS-CPS-data']
[21, 'P-PairDep-pvalue']
[22, 'P-PairDep-RelStats']
[23, 'T-PairDep-pvalue']
[24, 'T-PairDep-RelStats']
[25, 'coding_region']
[26, 'ref_cigar_s0b']
[27, 'ref_cigar']
